# Evaluate one model on one problem, end to endThis notebook does three things:1. pull the dataset from the Hub and pick a single row,2. ask a model to solve that row's question through OpenRouter,3. grade the reply with the **source repository's own** `parse_answer()` and `verify()`.Step 3 is the point. The dataset ships a gold `answer` column, but string-matching amodel's reply against it is the *wrong* grader: many families accept more than onevalid answer, so a correct solution can differ from the stored one character forcharacter. The family's `verify()` is the authority, and it is the same function thatgraded the dataset when it was built.### Prerequisites- **Read access to `guijinSON/arxiv-gv-problems`, which is a private repo.** The  generator modules live there, not on the Hub. Authenticate with `gh auth login`  (or have an SSH key configured) before running.- An OpenRouter key in the `OPENROUTER_API_KEY` environment variable. The notebook  never stores it in a cell.- `pip install huggingface_hub`

## 1. Configuration

In [ ]:
import os, sys, json, csv, random, subprocess, importlib.util, io, contextlib, urllib.request, urllib.error, timeHF_DATASET = "amphora/math-intuition-20260905-408-easy-30"CSV_NAME   = "math-intuition-20260905-408-easy-30.csv"GH_REPO    = "guijinSON/arxiv-gv-problems"REPO_DIR   = os.path.abspath("./arxiv-gv-problems")   # local clone targetMODEL      = "openai/gpt-5.6-terra"   # any OpenRouter model idEFFORT     = "medium"                 # reasoning effortMAX_TOKENS = 32000# None -> pick a random row. Set an integer to pin a specific one, or set# PICK_PAPER to target a family by arXiv id.ROW_INDEX  = NonePICK_PAPER = NoneSEED       = 0                        # controls which row is picked, not the problemcsv.field_size_limit(sys.maxsize)     # 60 rows have a question over the 128 KB defaultprint("config ok")

## 2. Get the generator repositoryThe graders are Python modules in the repo (`results/<arxiv_id>/gen_<id>.py`), notfiles on the Hub. About half of them import a shared `gvlib` from the repo root, andthey locate it *relative to their own path* — so the module has to stay inside a realclone. Downloading a single file will not work.

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):    print(f"cloning {GH_REPO} (private — needs your GitHub auth) ...")    r = subprocess.run(["gh", "repo", "clone", GH_REPO, REPO_DIR],                       capture_output=True, text=True)    if r.returncode != 0:                       # fall back to plain git        r = subprocess.run(["git", "clone", f"https://github.com/{GH_REPO}.git", REPO_DIR],                           capture_output=True, text=True)    if r.returncode != 0:        raise SystemExit(            "clone failed — this repo is private, so you need access.\n"            "Run `gh auth login`, or configure an SSH key, then re-run.\n\n"            + (r.stderr or "")[:600])    print("cloned.")else:    print(f"reusing existing clone at {REPO_DIR}")n_mod = len([d for d in os.listdir(os.path.join(REPO_DIR, "results"))             if os.path.isdir(os.path.join(REPO_DIR, "results", d))])print(f"{n_mod} result directories available")

## 3. Pull the dataset and pick one row

In [ ]:
from huggingface_hub import hf_hub_downloadpath = hf_hub_download(HF_DATASET, CSV_NAME, repo_type="dataset")   # cached after first runrows = list(csv.DictReader(open(path, encoding="utf-8")))print(f"{len(rows):,} rows, {len(set(r['paper'] for r in rows))} families")pool = [r for r in rows if r["paper"] == PICK_PAPER] if PICK_PAPER else rowsif not pool:    raise SystemExit(f"no rows for paper {PICK_PAPER!r}")row = pool[ROW_INDEX] if ROW_INDEX is not None else random.Random(SEED).choice(pool)print(f"\npicked  : {row['id']}")print(f"paper   : arXiv:{row['paper']}   preset={row['preset']}  seed={row['seed']}")print(f"domain  : {row['native_domain']} / {row['computational_core']}   track {row['track']}")print(f"search  : {row['search_space']}")print("\n--- question ---")print(row["question"][:1500] + ("\n... [truncated]" if len(row["question"]) > 1500 else ""))

## 4. Rebuild the instance, and prove the rebuild is faithful`verify(inst, answer)` needs the **instance object**, not the question string — it hasto re-derive the problem's internal structure to check a witness. The CSV carries the`seed` and `params` that produced the row, so the instance can be reconstructed exactly.The assertion below is the load-bearing part. If the rebuilt instance rendered to adifferent question than the one in the CSV, we would be grading the model's answeragainst a *different problem* and the verdict would be meaningless. Never skip it.

In [ ]:
def load_family(repo_dir, paper):    """Import results/<paper>/gen_*.py. Modules may print at import time; keep it quiet."""    import glob    hits = glob.glob(os.path.join(repo_dir, "results", paper, "gen_*.py"))    if not hits:        raise SystemExit(f"no generator for {paper} in {repo_dir}")    spec = importlib.util.spec_from_file_location(f"gen_{paper.replace('.', '_')}", hits[0])    mod  = importlib.util.module_from_spec(spec)    with contextlib.redirect_stdout(io.StringIO()):        spec.loader.exec_module(mod)    return mod, hits[0]fam, fam_path = load_family(REPO_DIR, row["paper"])print(f"loaded {os.path.relpath(fam_path, REPO_DIR)}")params = json.loads(row["params"])inst   = fam.make_instance(seed=int(row["seed"]), **params)assert fam.render(inst) == row["question"], (    "REBUILD MISMATCH — the reconstructed instance does not render to the question "    "in the dataset. Grading would compare against a different problem. Stop here.")print("integrity check passed: rebuilt instance renders to the exact dataset question")gold = json.loads(row["answer"])ok, why = fam.verify(inst, gold)print(f"sanity: the stored gold answer verifies -> {ok} ({why})")

## 5. Ask the model

In [ ]:
KEY = os.environ.get("OPENROUTER_API_KEY")if not KEY:    raise SystemExit("set OPENROUTER_API_KEY in your environment first (not in this notebook)")def ask(model, prompt, key, effort=EFFORT, max_tokens=MAX_TOKENS, timeout=1800):    """One OpenRouter call. Mirrors scripts/harden.py: the rendered question is sent    verbatim as the whole user message — no extra framing, no answer-format coaching,    because the family's render() already states the required format."""    body = json.dumps({        "model": model,        "reasoning": {"effort": effort},        "messages": [{"role": "user", "content": prompt}],        "max_tokens": max_tokens,    }).encode()    req = urllib.request.Request(        "https://openrouter.ai/api/v1/chat/completions", data=body,        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"})    try:        with urllib.request.urlopen(req, timeout=timeout) as resp:            payload = json.loads(resp.read().decode())    except urllib.error.HTTPError as e:        raise SystemExit(f"http {e.code}: {e.read()[:400].decode('utf-8','replace')}")    choice = payload["choices"][0]    return choice["message"]["content"], choice.get("finish_reason"), payload.get("usage", {})t0 = time.time()reply, finish, usage = ask(MODEL, row["question"], KEY)print(f"{MODEL} — {time.time()-t0:.1f}s, finish={finish}, usage={usage}")print("\n--- reply (tail) ---")print((reply or "")[-1200:])

## 6. Grade it with the repo's own codeTwo steps, and they fail for different reasons:- `parse_answer(text)` returns `None` when it cannot find a well-formed answer in the  reply. That is **not** the same as a wrong answer — it usually means the model  ignored the required output format.- `verify(inst, answer)` returns `(bool, reason)`. This is the authority.

In [ ]:
answer = fam.parse_answer(reply or "")if answer is None:    correct, reason = False, "parse_answer returned None — no well-formed answer in the reply"else:    correct, reason = fam.verify(inst, answer)print("=" * 66)print(f"  problem   : {row['id']}  (arXiv:{row['paper']}, {row['preset']})")print(f"  model     : {MODEL}")print(f"  parsed    : {answer is not None}")print(f"  CORRECT   : {correct}")print(f"  reason    : {reason}")print("=" * 66)if answer is not None:    same = json.dumps(answer, default=str, sort_keys=True) == json.dumps(gold, default=str, sort_keys=True)    print(f"\n  identical to the stored gold answer: {same}")    if correct and not same:        print("  -> correct but different: this family accepts multiple valid answers,")        print("     which is exactly why verify() is the grader and string equality is not.")

## Notes**This is the easy rung.** Every row here is each family's easiest preset, where thegenerator-verifier gap is narrowest. A model doing well on this slice tells you littleabout the hard presets.**To evaluate at scale**, loop over rows and reuse the loaded module per family —importing is the slow part, and some generators take seconds per instance. Group by`paper` so each module is imported once.**To evaluate uncontaminated**, do not use this file at all: the answers are public.Reconstruct fresh instances straight from the generators with unseen seeds —`fam.make_instance(seed=<new>, **fam.DIFFICULTY['hard'])` — and grade the same way.